In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType

In [0]:
df = spark.table('workspace.bronze.crm_cust_info')

In [0]:
df.display()

# Data transformation - TO DO list after exploring the data
  * ~~Rename header/column names to a much friendly names~~
  * ~~trimming whitespaces for all string type columns~~
  * ~~Normalize marital status abbreviation to a friendly name from 'M' to 'Maried' & 'S' to 'Single'~~
  * ~~Normalize the Gender abbreviation to a friendly name from 'M' to 'Male' & 'F' to'Female'~~
  * ~~column data type are ok~~
  * ~~there are no NULL values~~
  * ~~there are no empty values~~

In [0]:

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

df.display()

In [0]:
df = (
    df
    .withColumn('cst_marital_status',
                F.when(F.upper(F.col('cst_marital_status')) == 'S', 'Single')
                .when(F.upper(F.col('cst_marital_status')) == 'M', 'Married')
                .otherwise('n/a')
                )
    .withColumn('cst_gndr',
                F.when(F.upper(F.col('cst_gndr')) == 'M', 'Male')
                .when(F.upper(F.col('cst_gndr')) == 'F', 'Female')
                .otherwise('n/a')
                )
    )

df.display()

In [0]:
RENAME_MAP = {
    'cst_id': 'customer_id',
    'cst_key': 'customer_key',
    'cst_firstname': 'first_name',
    'cst_lastname': 'last_name',
    'cst_marital_status': 'marital_status',
    'cst_gndr': 'gender',
    'cst_create_date': 'created_date'
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

df.display()

# Write into Silver Layer

In [0]:
(
  df.write
    .mode('overwrite')
    .format('delta')
    .saveAsTable('workspace.silver.crm_customers')
)

In [0]:
%sql
SELECT
*
FROM workspace.silver.crm_customers